In [5]:
%load_ext Cython


In [7]:
def voronoi_finite_polygons_2d(vor, radius=None):
    """
    Reconstruct infinite voronoi regions in a 2D diagram to finite regions.
    
    Parameters
    ----------
    vor : Voronoi
        Input diagram
    radius : float, optional
        Distance to 'points at infinity'.
    
    Returns
    -------
    regions : list of lists
        Indices of vertices in each revised Voronoi regions.
        Vertices are sorted counter-clockwise.
    vertices : list of lists
        Coordinates for revised Voronoi vertices. Same as coordinates
        of input vertices, with 'points at infinity' appended to the
        end.
    
    """
    
    if vor.points.shape[1] != 2:
        raise ValueError("Requires 2D input")
    
    new_regions = []
    new_vertices = vor.vertices.tolist()
    
    center = vor.points.mean(axis=0)
    if radius is None:
        radius = vor.points.ptp().max()*2
    
    # Construct a map containing all ridges for a given point
    all_ridges = {}
    for (p1, p2), (v1, v2) in zip(vor.ridge_points, vor.ridge_vertices):
        all_ridges.setdefault(p1, []).append((p2, v1, v2))
        all_ridges.setdefault(p2, []).append((p1, v1, v2))
    
    # Reconstruct infinite regions
    for p1, region in enumerate(vor.point_region):
        vertices = vor.regions[region]
    
        if all(v >= 0 for v in vertices):
            # finite region
            new_regions.append(vertices)
            continue
    
        # reconstruct a non-finite region
        ridges = all_ridges[p1]
        new_region = [v for v in vertices if v >= 0]
    
        for p2, v1, v2 in ridges:
            if v2 < 0:
                v1, v2 = v2, v1
            if v1 >= 0:
                # finite ridge: already in the region
                continue
    
            # Compute the missing endpoint of an infinite ridge
    
            t = vor.points[p2] - vor.points[p1] # tangent
            t /= np.linalg.norm(t)
            n = np.array([-t[1], t[0]])  # normal
    
            midpoint = vor.points[[p1, p2]].mean(axis=0)
            direction = np.sign(np.dot(midpoint - center, n)) * n
            far_point = vor.vertices[v2] + direction * radius
    
            new_region.append(len(new_vertices))
            new_vertices.append(far_point.tolist())
    
    
        # sort region counterclockwise
        vs = np.asarray([new_vertices[v] for v in new_region])
        c = vs.mean(axis=0)
        angles = np.arctan2(vs[:,1] - c[1], vs[:,0] - c[0])
        new_region = np.array(new_region)[np.argsort(angles)]
    
        # finish
        new_regions.append(new_region.tolist())
    
    return new_regions, np.asarray(new_vertices)

In [8]:
boxsize = 10

In [10]:
import numpy as np
from scipy.spatial import Voronoi, ConvexHull


In [12]:
points = np.random.rand(100,2) * boxsize
vor = Voronoi(points)
regions, vertices = voronoi_finite_polygons_2d(vor, boxsize)

In [13]:
def get_polygons_properties_2d(regions, vertices):
    """
    Extract the perimeter and area for each finite polygon in 2D
    
    Parameters
    -----
    regions : list of lists
        Indices of vertices in each revised Voronoi regions.
    vertices : list of lists
        Coordinates for revised Voronoi vertices. Same as coordinates
        of input vertices, with 'points at infinity' appended to the
        end.
    
    Returns
    -----
    properties_index : list
        List of index for each region in regions treated (probably not useful since perimeter and area are
        already acquired in the order of region in regions)
    properties_perimeter : list of float
        Perimeter of each region in regions treated
    properties_area : list of float
        Area of each region in regions treated
        Warning : In 2D : ConvexHull.area is the perimeter, while ConvexHull.volume is the area
    """
    
    properties_index = []
    properties_perimeter = []
    properties_area = []
    
    
    index = 0
    for index, region in enumerate(regions):
        polygon = vertices[region]
        properties_index.append(index)
        properties_perimeter.append(ConvexHull(polygon).area)
        properties_area.append(ConvexHull(polygon).volume)
    
    return properties_index,properties_perimeter,properties_area

In [14]:
%timeit get_polygons_properties_2d(regions, vertices)

191 ms ± 4.9 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [17]:
%%cython -a
import numpy as np
from scipy.spatial import Voronoi, ConvexHull
def get_polygons_properties_2d_cython(regions, vertices):
    """
    Extract the perimeter and area for each finite polygon in 2D
    
    Parameters
    -----
    regions : list of lists
        Indices of vertices in each revised Voronoi regions.
    vertices : list of lists
        Coordinates for revised Voronoi vertices. Same as coordinates
        of input vertices, with 'points at infinity' appended to the
        end.
    
    Returns
    -----
    properties_index : list
        List of index for each region in regions treated (probably not useful since perimeter and area are
        already acquired in the order of region in regions)
    properties_perimeter : list of float
        Perimeter of each region in regions treated
    properties_area : list of float
        Area of each region in regions treated
        Warning : In 2D : ConvexHull.area is the perimeter, while ConvexHull.volume is the area
    """
    
    properties_index = []
    properties_perimeter = []
    properties_area = []
    
    
    index = 0
    for index, region in enumerate(regions):
        polygon = vertices[region]
        properties_index.append(index)
        properties_perimeter.append(ConvexHull(polygon).area)
        properties_area.append(ConvexHull(polygon).volume)
    
    return properties_index,properties_perimeter,properties_area

In [19]:
%timeit get_polygons_properties_2d_cython(regions, vertices)

235 ms ± 38.7 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [20]:
%%cython -a
import numpy as np
from scipy.spatial import Voronoi, ConvexHull
def get_polygons_properties_2d_cython_improved(regions, vertices):
    """
    Extract the perimeter and area for each finite polygon in 2D
    
    Parameters
    -----
    regions : list of lists
        Indices of vertices in each revised Voronoi regions.
    vertices : list of lists
        Coordinates for revised Voronoi vertices. Same as coordinates
        of input vertices, with 'points at infinity' appended to the
        end.
    
    Returns
    -----
    properties_index : list
        List of index for each region in regions treated (probably not useful since perimeter and area are
        already acquired in the order of region in regions)
    properties_perimeter : list of float
        Perimeter of each region in regions treated
    properties_area : list of float
        Area of each region in regions treated
        Warning : In 2D : ConvexHull.area is the perimeter, while ConvexHull.volume is the area
    """
    
    properties_index = []
    properties_perimeter = []
    properties_area = []
    
    
    index = 0
    len_regions = len(regions)
    for index in range(len_regions):
        polygon = vertices[regions[index]]
        properties_index.append(index)
        convHull = ConvexHull(polygon)
        properties_perimeter.append(convHull.area)
        properties_area.append(convHull.volume)
    
    return properties_index,properties_perimeter,properties_area

In [21]:
%timeit get_polygons_properties_2d_cython(regions, vertices)

204 ms ± 41.6 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [48]:
type(vertices)
print(np.shape(vertices))
print(vertices[0,1])

(209, 2)
0.18001354547528636


In [85]:
regions[2]

[66, 29, 28, 27, 63]

In [93]:
vertices[66]


array([-1.72483266,  4.04850392])

In [89]:
vertices[[66, 29, 28, 27, 63]] == vertices[regions[2]]

array([[ True,  True],
       [ True,  True],
       [ True,  True],
       [ True,  True],
       [ True,  True]])

In [98]:
ConvexHull

scipy.spatial.qhull.ConvexHull

In [151]:
%%cython -a
import numpy as np
from scipy.spatial import Voronoi, ConvexHull
# cimport cython
# @cython.boundscheck(False)  # Deactivate bounds checking
# @cython.wraparound(False)   # Deactivate negative indexing.
def get_polygons_properties_2d_cython_typed(regions, vertices):
    """
    Extract the perimeter and area for each finite polygon in 2D
    
    Parameters
    -----
    regions : list of lists
        Indices of vertices in each revised Voronoi regions.
    vertices : list of lists
        Coordinates for revised Voronoi vertices. Same as coordinates
        of input vertices, with 'points at infinity' appended to the
        end.
    
    Returns
    -----
    properties_index : list
        List of index for each region in regions treated (probably not useful since perimeter and area are
        already acquired in the order of region in regions)
    properties_perimeter : list of float
        Perimeter of each region in regions treated
    properties_area : list of float
        Area of each region in regions treated
        Warning : In 2D : ConvexHull.area is the perimeter, while ConvexHull.volume is the area
    """
    
    ## Typing variables
    cdef int len_regions = len(regions)
    cdef Py_ssize_t index
    
    properties_index_np = np.zeros(len_regions, dtype=np.int)
    properties_perimeter_np = np.zeros(len_regions, dtype=np.float64)
    properties_area_np = np.zeros(len_regions, dtype=np.float64)
    
    cdef int[:] properties_index = properties_index_np
    cdef double[:] properties_perimeter = properties_perimeter_np
    cdef double[:] properties_area = properties_area_np
    cdef double convHull_area
    cdef double convHull_volume
    # cdef double[:,:] polygon

    for index in range(len_regions):
        polygon = vertices[regions[index]]
        properties_index[index] = index
        convHull = ConvexHull(polygon)
        convHull_area = convHull.area
        convHull_volume = convHull.volume
        properties_perimeter[index] = convHull_area
        properties_area[index] = convHull_volume
        
    return properties_index,properties_perimeter,properties_area

In [154]:
pi_1, pp_1, pa_1 = get_polygons_properties_2d(regions, vertices)

In [155]:
pi_2, pp_2, pa_2 = get_polygons_properties_2d_cython_typed(regions, vertices)

In [156]:
pi_1 == pi_2

False

In [168]:
for x,y in zip(pa_1,pa_2):
    # print("x is {}, y is {}".format(x,y))
    print(x == y)

True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True


<MemoryView of 'ndarray' at 0x1ff6d465890>